In [53]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [54]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,f1_score,r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer

In [55]:
#Loading dataset
df=pd.read_csv("car.csv")
df.head(5)

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


In [56]:
df.shape

(4340, 8)

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   name           4340 non-null   object
 1   year           4340 non-null   int64 
 2   selling_price  4340 non-null   int64 
 3   km_driven      4340 non-null   int64 
 4   fuel           4340 non-null   object
 5   seller_type    4340 non-null   object
 6   transmission   4340 non-null   object
 7   owner          4340 non-null   object
dtypes: int64(3), object(5)
memory usage: 271.4+ KB


In [58]:
#extracting brand name from the name col
df['brand'] = df['name'].apply(lambda x: x.split(' ')[0])
df.drop(columns='name',axis=1,inplace=True)

In [59]:
df.describe()

,year,selling_price,km_driven
count,4340.000000,4.340000e+03,4340.000000
mean,2013.090783,5.041273e+05,66215.777419
std,4.215344,5.785487e+05,46644.102194
min,1992.000000,2.000000e+04,1.000000
25%,2011.000000,2.087498e+05,35000.000000
50%,2014.000000,3.500000e+05,60000.000000
75%,2016.000000,6.000000e+05,90000.000000
max,2020.000000,8.900000e+06,806599.000000


In [60]:
num_col=['year','km_driven']
cat_col=['fuel','seller_type','transmission','owner','brand']


In [61]:
for col in cat_col:
  print(df[col].value_counts())
  print()

fuel
Diesel      2153
Petrol      2123
CNG           40
LPG           23
Electric       1
Name: count, dtype: int64

seller_type
Individual          3244
Dealer               994
Trustmark Dealer     102
Name: count, dtype: int64

transmission
Manual       3892
Automatic     448
Name: count, dtype: int64

owner
First Owner             2832
Second Owner            1106
Third Owner              304
Fourth & Above Owner      81
Test Drive Car            17
Name: count, dtype: int64

brand
Maruti           1280
Hyundai           821
Mahindra          365
Tata              361
Honda             252
Ford              238
Toyota            206
Chevrolet         188
Renault           146
Volkswagen        107
Skoda              68
Nissan             64
Audi               60
BMW                39
Fiat               37
Datsun             37
Mercedes-Benz      35
Mitsubishi          6
Jaguar              6
Land                5
Volvo               4
Ambassador          4
Jeep                3
Ope

In [62]:
#removing rare_brand to others
brand_counts = df['brand'].value_counts()

rare_brands = brand_counts[brand_counts < 10].index

df['brand'] = df['brand'].replace(rare_brands, 'Other')

In [63]:
df['brand'].value_counts()

,count
brand,
Maruti,1280
Hyundai,821
Mahindra,365
Tata,361
Honda,252
Ford,238
Toyota,206
Chevrolet,188
Renault,146


In [64]:
df.head()

,year,selling_price,km_driven,fuel,seller_type,transmission,owner,brand
0,2007,60000,70000,Petrol,Individual,Manual,First Owner,Maruti
1,2007,135000,50000,Petrol,Individual,Manual,First Owner,Maruti
2,2012,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai
3,2017,250000,46000,Petrol,Individual,Manual,First Owner,Datsun
4,2014,450000,141000,Diesel,Individual,Manual,Second Owner,Honda


In [65]:
# encoding fuel,seller_type and owner
df=pd.get_dummies(df,columns=['fuel','seller_type','owner','brand'],dtype='int',drop_first=True)

In [66]:
df.head(5)

,year,selling_price,km_driven,transmission,fuel_Diesel,fuel_Electric,fuel_LPG,fuel_Petrol,seller_type_Individual,seller_type_Trustmark Dealer,...,brand_Mahindra,brand_Maruti,brand_Mercedes-Benz,brand_Nissan,brand_Other,brand_Renault,brand_Skoda,brand_Tata,brand_Toyota,brand_Volkswagen
0,2007,60000,70000,Manual,0,0,0,1,1,0,...,0,1,0,0,0,0,0,0,0,0
1,2007,135000,50000,Manual,0,0,0,1,1,0,...,0,1,0,0,0,0,0,0,0,0
2,2012,600000,100000,Manual,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,2017,250000,46000,Manual,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,2014,450000,141000,Manual,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [67]:
# encoding Transmission
df['transmission']=df['transmission'].map({'Manual':1,'Automatic':0})

In [68]:
#Feature engineering
df['car_age']=2026-df['year']

In [69]:
df.drop(columns='year',axis=1,inplace=True)

In [70]:
df.head(5)

,selling_price,km_driven,transmission,fuel_Diesel,fuel_Electric,fuel_LPG,fuel_Petrol,seller_type_Individual,seller_type_Trustmark Dealer,owner_Fourth & Above Owner,...,brand_Maruti,brand_Mercedes-Benz,brand_Nissan,brand_Other,brand_Renault,brand_Skoda,brand_Tata,brand_Toyota,brand_Volkswagen,car_age
0,60000,70000,1,0,0,0,1,1,0,0,...,1,0,0,0,0,0,0,0,0,19
1,135000,50000,1,0,0,0,1,1,0,0,...,1,0,0,0,0,0,0,0,0,19
2,600000,100000,1,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,14
3,250000,46000,1,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,9
4,450000,141000,1,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,12


In [71]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['selling_price'],axis=1),
                                                    df['selling_price'],
                                                    test_size=0.2,
                                                    random_state=42)

In [72]:
X_train

,km_driven,transmission,fuel_Diesel,fuel_Electric,fuel_LPG,fuel_Petrol,seller_type_Individual,seller_type_Trustmark Dealer,owner_Fourth & Above Owner,owner_Second Owner,...,brand_Maruti,brand_Mercedes-Benz,brand_Nissan,brand_Other,brand_Renault,brand_Skoda,brand_Tata,brand_Toyota,brand_Volkswagen,car_age
227,20000,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,9
964,50000,1,1,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,8
2045,25000,1,0,0,0,1,1,0,0,1,...,1,0,0,0,0,0,0,0,0,13
1025,70000,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,15
4242,72000,1,1,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3444,50000,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,20
466,80000,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,15
3092,51000,1,1,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,10
3772,80000,1,1,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,13


In [73]:
y_train

,selling_price
227,1500000
964,500000
2045,92800
1025,95000
4242,685000
...,...
3444,500000
466,133000
3092,665000
3772,250999


In [74]:
# Model building and Training
rf=RandomForestRegressor(n_estimators=400,
                         max_depth=None,
                         random_state=42)
rf.fit(X_train,y_train)


RandomForestRegressor(n_estimators=400, random_state=42)

In [75]:
# prediction
y_pred=rf.predict(X_test)

In [76]:
# Accuracy Score
accuracy=r2_score(y_test,y_pred)
print(accuracy)


0.6324374159537822


In [78]:
import joblib
joblib.dump(rf, "car_price_rf.pkl")


['car_price_rf.pkl']